In [1]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [2]:
 def load_pdf_files(data):
     loader = DirectoryLoader(
         data,
         glob="*.pdf",
         loader_cls=PyPDFLoader
     )

     documents = loader.load()
     return documents

In [3]:
extracted_data = load_pdf_files(r"C:\Users\DELL\Documents\Projects\Medical-Chatbot\Medical-Chatbot\data")

In [4]:
len(extracted_data)

637

In [5]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimal_docs

In [6]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [7]:
def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    texts_chunk=text_splitter.split_documents(minimal_docs)
    return texts_chunk

In [8]:
texts_chunk=text_split(minimal_docs)
print(f"Number of chunks: {len(texts_chunk)}")

Number of chunks: 5860


In [9]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )
    return embeddings
embedding = download_embeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
embedding

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [11]:
vector = embedding.embed_query("Hello world")

In [12]:
print("Vector Length:", len(vector))

Vector Length: 384


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY


In [15]:
from pinecone import Pinecone, ServerlessSpec
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [16]:
pc

In [17]:
index_name = "medi-bot"

In [18]:
from langchain_pinecone import PineconeVectorStore

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    # Only upload documents if index was just created
    docsearch = PineconeVectorStore.from_documents(
        documents=texts_chunk,
        embedding=embedding,
        index_name=index_name
    )
else:
    # Index already exists, just connect to it
    docsearch = PineconeVectorStore.from_existing_index(
        index_name=index_name,
        embedding=embedding
    )
    
index = pc.Index(index_name)

In [19]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [20]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='142a5d79-76d0-48aa-aab5-1467dd304350', metadata={'source': 'C:\\Users\\DELL\\Documents\\Projects\\Medical-Chatbot\\Medical-Chatbot\\data\\Medibook.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='0ab3a6ba-3d1a-4d1c-b1a0-c0d7ea8321a9', metadata={'source': 'C:\\Users\\DELL\\Documents\\Projects\\Medical-Chatbot\\Medical-Chatbot\\data\\Medibook.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='5ccafbff-c2b3-46f2-b6ea-b1ce300b9d08', metadata={'source': 'C:\\Users\\DELL\\Documents\\Projects\\Medical-Chatbot\\Medical-Chatbot\\data\\Medibook.pdf'}, page_content='Acidosis see Respiratory acidosis; 

In [41]:
from langchain_google_genai import ChatGoogleGenerativeAI

chatModel = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [42]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [43]:
system_prompt = (
    "You are a Medical assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "the question. If you don't know the answer, say that you"
    "don't know. Use three sentences maximum and keep the"
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [44]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [45]:
response = rag_chain.invoke({"input":"what is Acromegaly and gigantism?"})
print(response["answer"])

Acromegaly is a disorder where the pituitary gland abnormally releases a chemical, causing increased growth in bone and soft tissue after bone growth has stopped. If this abnormality occurs before bone growth stops, it results in unusual height, a condition known as gigantism.


In [46]:
response = rag_chain.invoke({"input":"Acne?"})
print(response["answer"])

Acne is a general term for a skin disorder characterized by inflamed sebaceous glands. It develops when pores or hair follicles become blocked, leading to the collection of sebum, a waxy material. This blockage can result in small swellings on the skin, and the accumulation of bacteria and dead skin cells can cause further inflammation.
